In [1]:
# NUMERICAL & DATA MANIPULATION

import pandas as pd
import numpy as np

# DATA VISUALIZATION

import matplotlib.pyplot as plt
import seaborn as sns

### **Data import**

In [2]:
# CREATE A DATA FRAME FROM EACH CSV FILE IN THE DATA FOLDER

customers = pd.read_csv('data/olist_customers_dataset.csv')
order_items = pd.read_csv('data/olist_order_items_dataset.csv')
order_payments = pd.read_csv('data/olist_order_payments_dataset.csv')
orders = pd.read_csv('data/olist_orders_dataset.csv')
products = pd.read_csv('data/olist_products_dataset.csv')


#### **1. EDA**

The content of each single dataset is explored to ensure consistency and spot possible issues.

##### **Customers dataset**

The customers dataset is ready for use. All data formats are appropriate, and no null values are present.

In [3]:
# DISPLAY TOP TWO ROWS IN THE DATAFRAME

customers.head(2)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [4]:
# INSPECT DATAFRAME MEDTADATA INCLUDING ENTRIES, COLUMNS, NON-NULL COUNTS, AND DATA TYPES.

customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [5]:
# CHECK IF THE DATAFRAME CONTAINS DUPLICATED DATA.

print(f'The customer dataframe has {customers.duplicated().sum()} duplicated entries.')

The customer dataframe has 0 duplicated entries.


In [6]:
# DISPLAY THE NUMBER OF UNIQUE VALUES PER COLUMN

customers.nunique(dropna=False)

customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

The number of unique values in the customer_id and customer_unique_id do not match.
This discrepancy is expected, as a new customer_id is assigned to each individual purchase made by the same unique customer (customer_unique_id), resulting in duplicated values in the customer_unique_id column.
That mean customer_id is the primary key of the data frame, which also appears as a foreign key in other frames. Using customer_id ensures that all orders can be merged correctly and consistently mapped across datasets.

In [7]:
# DISPLAY THE NUMBER OF UNIQUE VALUES (FALSE) AND DUPLICATED VALUES (TRUE) IN THE CUSTOMER_UNIQUE_ID

customers['customer_unique_id'].duplicated().value_counts()

customer_unique_id
False    96096
True      3345
Name: count, dtype: int64

##### **Order items dataset**

The order items dataset is ready for use. All data formats are appropriate, and no null values are present.

In [8]:
# DISPLAY TOP TWO ROWS IN THE DATAFRAME

order_items.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


In [9]:
# INSPECT DATAFRAME MEDTADATA INCLUDING ENTRIES, COLUMNS, NON-NULL COUNTS, AND DATA TYPES.

order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [10]:
# CHECK IF THE DATAFRAME CONTAINS DUPLICATED DATA.

print(f'The order items dataframe has {order_items.duplicated().sum()} duplicated entries.')

The order items dataframe has 0 duplicated entries.


The number of unique values in the order_id column (98,666) is lower than the total number of records in the orders DataFrame (99,491). This discrepancy is likely due to orders that were canceled, recently placed, or are still in an initial status.

This issue will be addressed once the Data frames are merged and the order lifecycle is analyzed more comprehensively.

In [11]:
# DISPLAY THE NUMBER OF UNIQUE VALUES PER COLUMN

order_items.nunique(dropna= False)

order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

##### **Orders dataset**

The orders dataset requires special attention in the following areas:

* **Date and time fields** are incorrectly formatted as objects and need to be converted to proper date or datetime formats.
* **Missing (null) values** are present; a more in-depth exploratory analysis will be conducted to determine the appropriate handling strategy for these records.

In [12]:
# DISPLAY TOP TWO ROWS IN THE DATAFRAME

orders.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


In [13]:
# INSPECT DATAFRAME MEDTADATA INCLUDING ENTRIES, COLUMNS, NON-NULL COUNTS, AND DATA TYPES.

orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [14]:
# CHECK IF THE DATAFRAME CONTAINS DUPLICATED DATA.

print(f'The orders dataframe has {orders.duplicated().sum()} duplicated entries.')

The orders dataframe has 0 duplicated entries.


**Handling wrong data types**

In [15]:
# LIST OF COLUMNS THAT SHOULD BE DATETIME BUT CURRENTLY HAVE THE WRONG DATA TYPE.

dates = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


# APPLY CORRECT DATA TYPE TO THE LISTED COLUMNS

orders[dates] = orders[dates].apply(pd.to_datetime, errors= 'coerce')

In [16]:
# CHECK THE NEW DATA TYPES HAVE BEEN CONVERTED PROPERLY

orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

**Handling null values**

Intuitively, some of the missing values are expected.
For example, orders may not have been approved yet, delivered to the carrier, or received by the customer.
Let’s investigate this in more detail.

In [17]:
# DISPLAY NUMBER OF NULL VALUES PER COLUMN

orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [18]:
# GENERATE A DATA FRAME WITH ALL ENTRIES CONTAINING AT LEAST A NULL VALUE

orders_nulls = orders[orders.isnull().any(axis=1)]

# DISPLAY TOP 3 ROWS OF THE NEWLY CREATED MISSING_DATA FRAME

orders_nulls.head(3)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaT,2018-06-28
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaT,NaT,2018-08-21


In [19]:
# DISTRIBUTION OF NULLS BY ORDER STATUS

orders_nulls['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered        23
created           5
approved          2
Name: count, dtype: int64

**'orders_approved_at' missing values**

Most of the missing values are expected, as they correspond to orders that were either canceled or recently created.
Only a small fraction of orders progressed through the entire process without a recorded order_approved_at date. These cases could be monitored to ensure data accuracy and consistency. However, they are not critical at this stage, since all of these orders have already been delivered.

In [20]:
# DISPLAY THE DISTRIBUTION OF NULLS IN THE COLUMNS ORDERS_APPROVED_AT BY ORDER_STATUS

orders_nulls[orders_nulls['order_approved_at'].isnull()]['order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

**order_delivered_carrier_date missing values**

In [21]:
# DISPLAY THE DISTRIBUTION OF NULLS IN THE COLUMNS ORDERS_DELIVERED_CARRIER_DATE BY ORDER_STATUS

orders_nulls[orders_nulls['order_delivered_carrier_date'].isnull()]['order_status'].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

**order_delivered_customer_date missing values**

In [22]:
# DISPLAY THE DISTRIBUTION OF NULLS IN THE COLUMNS ORDERS_DELIVERED_CUSTOMER_DATE BY ORDER_STATUS

orders_nulls[orders_nulls['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [23]:
# DISPLAY THE NUMBER OF UNIQUE VALUES PER COLUMN

orders.nunique(dropna= False)

order_id                         99441
customer_id                      99441
order_status                         8
order_purchase_timestamp         98875
order_approved_at                90734
order_delivered_carrier_date     81019
order_delivered_customer_date    95665
order_estimated_delivery_date      459
dtype: int64

##### **Products dataset**

The products dataset is ready to be used. Formats make sense and there are no null values.

In [24]:
# DISPLAY TOP TWO ROWS IN THE DATAFRAME

products.head(2)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0


In [25]:
# INSPECT DATAFRAME MEDTADATA INCLUDING ENTRIES, COLUMNS, NON-NULL COUNTS, AND DATA TYPES.

products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [26]:
# CHECK IF THE DATAFRAME CONTAINS DUPLICATED DATA.

print(f'The product dataframe has {products.duplicated().sum()} duplicated entries.')

The product dataframe has 0 duplicated entries.


In [27]:
# DISPLAY THE NUMBER OF UNIQUE VALUES PER COLUMN

products.nunique(dropna= False)

product_id                    32951
product_category_name            74
product_name_lenght              67
product_description_lenght     2961
product_photos_qty               20
product_weight_g               2205
product_length_cm               100
product_height_cm               103
product_width_cm                 96
dtype: int64

### **Defining final data frames for analysis**

First, the most relevant KPIs will be defined to allow them to be merged and to ensure the most meaningful data is obtained.

In [28]:
# GENERATE THE FINAL FRAMES THAT WILL BE MERGED

customers_final = customers

orders_final = orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']]

order_items_final = order_items[['order_id', 'product_id', 'seller_id', 'order_item_id', 'price', 'freight_value']]

products_final = products[['product_id', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']]

### **1. Orders**

**a. Data frames adjustments**

In [29]:
# DEFINE AN AGGREGATED FRAME WITH THE RELEVENAT COLUMNS FROM THE ORDER_ITEMS DATA FRAME

agg_items = order_items_final.groupby(by= 'order_id').agg(
                                                                        number_items = ('product_id', 'count'),
                                                                        order_price = ('price', 'sum'),
                                                                        order_freight_value = ('freight_value', 'sum'))

# CREATE A COPY OF THE ORDERS_FINAL DATA FRAME TO AVOID CONTAMINATION WHEN RUNNING THE MERGING LOOP

orders_final_copy = orders_final.copy()

**b. Mergin the frames**

In [30]:
# LIST THE DATA FRAMES AND JOIN KEYS USE TO MERGE

orders_mergin_parameters = [(customers_final, 'customer_id'), (agg_items, 'order_id')]

# RUN A LOOP TO MERGE ALL DATA FRAMES

for frame, order_join_key in orders_mergin_parameters:
    
    df_orders = pd.merge(left= orders_final_copy, right= frame, how= 'left', on= order_join_key)
    orders_final_copy = df_orders
    
# DISPLAY THE TOP 3 ROWS OF THE NEWLY MERGED DATA FRAME

df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,number_items,order_price,order_freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,159.90,19.22


**c. Feature engeeniring**

In [31]:
# ADD GMV BEFORE CANCELLATIONS AND FAILED DELIVERIES TO THE AGG-ITEMS FRAME

df_orders['gmv_bcanc_and_fails'] = df_orders['order_price'].fillna(0) + df_orders['order_freight_value'].fillna(0)


# ADD A FLAG IF ORDERS HAVE BEEN CANCELED, HAS FAILED DELIVERY; OR IT HAS BEN SUCCESFULLLY DELIVERED

df_orders['is_canceled'] = df_orders['order_status'] == 'canceled'
df_orders['is_failed_delivery'] = df_orders['order_status'] == 'unavailable'
df_orders['is_delivered'] = df_orders['order_status'] == 'delivered'

# ADD A COLUMN WITH THE REAL REVENUE 

df_orders['price_aft_canc_and_fail'] = np.where(df_orders['is_delivered'], df_orders['order_price'], 0)

# ADD COLUMNS WITH THE TOTAL REVENUE LOSS AND THE LOSSES CAUSED BY CANCELLATIONS AND FAILED DELIVERIES

df_orders['price_loss_canc'] = np.where(df_orders['is_canceled'], df_orders['order_price'], 0)
df_orders['price_loss_fail'] = np.where(df_orders['is_failed_delivery'], df_orders['order_price'], 0)
df_orders['price_loss'] = df_orders['price_loss_canc'] + df_orders['price_loss_fail']

In [32]:
# DISPLAY THE TOP 3 ROWS OF THE DATA FRAME

df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,...,order_price,order_freight_value,gmv_bcanc_and_fails,is_canceled,is_failed_delivery,is_delivered,price_aft_canc_and_fail,price_loss_canc,price_loss_fail,price_loss
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,...,29.99,8.72,38.71,False,False,True,29.99,0.0,0.0,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,...,118.70,22.76,141.46,False,False,True,118.70,0.0,0.0,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,...,159.90,19.22,179.12,False,False,True,159.90,0.0,0.0,0.0


**d. Final checks**

In [33]:
# PROVIDE INFORMATION ABOUT THE NUMBER OF COLUMNS AND ROWS IN THE DATA FRAME

print(f'The df_items data frame contains {df_orders.shape[1]} columns, {df_orders.shape[0]} entries, and {df_orders.duplicated().sum()} duplicates.')

The df_items data frame contains 22 columns, 99441 entries, and 0 duplicates.


In [34]:
# INSPECT DATAFRAME MEDTADATA INCLUDING ENTRIES, COLUMNS, NON-NULL COUNTS, AND DATA TYPES.

df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 5   order_delivered_customer_date  96476 non-null  datetime64[ns]
 6   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 7   customer_unique_id             99441 non-null  object        
 8   customer_zip_code_prefix       99441 non-null  int64         
 9   customer_city                  99441 non-null  object        
 10  customer_state                 99441 non-null  object        
 11  number_items   

### **2. Ordered items**

The most relevant KPIs are extracted from the order_items DataFrame and merged with KPIs from other sources, such as product category, order status, and order purchase timestamp.

**a. Data frames adjustments**

In [35]:
# GENERATE A COPY OR THE ORDER_ITEMS_FINAL TO GUARANTEED DATA CONSISTENCY

df_items = order_items_final.copy()

**b. Merging the frames**

In [36]:
# MERGE DATA FRAMES STEP BY STEP

df_items = df_items.merge(products_final, on='product_id', how='left', validate='many_to_one')
df_items = df_items.merge(orders_final, on='order_id', how='left', validate='many_to_one')

**c. Feature engeeniring**

In [37]:
# DROP COLUMN CUSTOMER_ID AS IT LACKS IN RELEVANCE FOR THIS DATA FRAME

df_items = df_items.drop(columns=['customer_id', 'seller_id'], errors= 'ignore')

# ADD COLUMNS A FLAG IF ORDERS HAVE BEEN CANCELED, HAS FAILED DELIVERY; OR IT HAS BEN SUCCESFULLLY DELIVERED

df_items['is_canceled'] = df_items['order_status'] == 'canceled'
df_items['is_failed_delivery'] = df_items['order_status'] == 'unavailable'
df_items['is_delivered'] = df_items['order_status'] == 'delivered'


In [38]:
# DISPLAY THE TOP 3 ROWS OF DATA FRAME

df_items.head(3)

,order_id,product_id,order_item_id,price,freight_value,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_canceled,is_failed_delivery,is_delivered
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,1,58.9,13.29,cool_stuff,650.0,28.0,9.0,14.0,delivered,2017-09-13 08:59:02,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,False,False,True
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,1,239.9,19.93,pet_shop,30000.0,50.0,30.0,40.0,delivered,2017-04-26 10:53:06,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,False,False,True
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,1,199.0,17.87,moveis_decoracao,3050.0,33.0,13.0,33.0,delivered,2018-01-14 14:33:31,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,False,False,True


**d. Final checks**

In [39]:
# PROVIDE INFORMATION ABOUT THE NUMBER OF COLUMNS AND ROWS IN THE DATA FRAME

print(f'The df_items data frame contains {df_items.shape[1]} columns, {df_items.shape[0]} entries, and {df_items.duplicated().sum()} duplicates.')


The df_items data frame contains 18 columns, 112650 entries, and 0 duplicates.


In [40]:
# INSPECT DATAFRAME MEDTADATA INCLUDING ENTRIES, COLUMNS, NON-NULL COUNTS, AND DATA TYPES.

df_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  object        
 1   product_id                     112650 non-null  object        
 2   order_item_id                  112650 non-null  int64         
 3   price                          112650 non-null  float64       
 4   freight_value                  112650 non-null  float64       
 5   product_category_name          111047 non-null  object        
 6   product_weight_g               112632 non-null  float64       
 7   product_length_cm              112632 non-null  float64       
 8   product_height_cm              112632 non-null  float64       
 9   product_width_cm               112632 non-null  float64       
 10  order_status                   112650 non-null  object        
 11  

### **3. Customers**

**a. Data frames adjustments**

In [41]:
# GENERATE A COPY OR THE CUSTOMERS_FINAL AND THE DF_ORDERS FRAMES TO GUARANTEED DATA CONSISTENCY

customers_final_copy = customers_final[['customer_unique_id',
                      'customer_zip_code_prefix',
                      'customer_city',
                      'customer_state']].drop_duplicates('customer_unique_id')

df_orders_copy = df_orders.copy()

In [42]:
# DEFINE AN AGGREGATED FRAME WITH THE RELEVENAT COLUMNS FROM THE ORDER_ITEMS DATA FRAME

agg_orders = df_orders_copy.groupby(by= 'customer_unique_id').agg(
                                                                        n_orders = ('order_id', 'count'),
                                                                        total_price_bcanc_and_fail = ('order_price', 'sum'),
                                                                        total_price_aft_canc_and_fail = ('price_aft_canc_and_fail', 'sum'),
                                                                        total_price_loss_canc = ('price_loss_canc', 'sum'),
                                                                        total_price_loss_fail = ('price_loss_fail', 'sum'),
                                                                        total_price_loss = ('price_loss', 'sum'),
                                                                        total_freight_value = ('order_freight_value', 'sum'),
                                                                        avg_basket_value = ('order_price', 'mean'),
                                                                        n_cancellations = ('is_canceled', 'sum'),
                                                                        n_fail_delivery = ('is_failed_delivery', 'sum'),
                                                                        first_order_date = ('order_purchase_timestamp', 'min'),
                                                                        last_order_date = ('order_purchase_timestamp', 'max')
                                                                        )

**b. Merging the frames**

In [43]:
# MERGE AGG_ORDERS INTO CUSTOMERS_FINAL_COPY
    
df_customers = customers_final_copy.merge(right= agg_orders, how= 'left', on= 'customer_unique_id')
orders_final_copy = df_orders
    
# DISPLAY THE TOP 3 ROWS OF THE NEWLY MERGED DATA FRAME

df_customers.head(3)

,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,n_orders,total_price_bcanc_and_fail,total_price_aft_canc_and_fail,total_price_loss_canc,total_price_loss_fail,total_price_loss,total_freight_value,avg_basket_value,n_cancellations,n_fail_delivery,first_order_date,last_order_date
0,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,1,124.99,124.99,0.0,0.0,0.0,21.88,124.99,0,0,2017-05-16 15:05:35,2017-05-16 15:05:35
1,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,1,289.00,289.00,0.0,0.0,0.0,46.48,289.00,0,0,2018-01-12 20:48:24,2018-01-12 20:48:24
2,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1,139.94,139.94,0.0,0.0,0.0,17.79,139.94,0,0,2018-05-19 16:07:45,2018-05-19 16:07:45


**c. Feature engeeniring**

In [44]:
# ADD THE COLUMNS CANCELLATION_RATE AND FAILURE_RATE

df_customers['cancellation_rate']  =  df_customers['n_cancellations'] / df_customers['n_orders'].replace(0, np.nan)
df_customers['failure_rate'] = df_customers['n_fail_delivery'] / df_customers['n_orders'].replace(0, np.nan)


In [45]:
# DISPLAY THE TOP 3 ROWS OF DATA FRAME

df_customers.head(3)

,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,n_orders,total_price_bcanc_and_fail,total_price_aft_canc_and_fail,total_price_loss_canc,total_price_loss_fail,total_price_loss,total_freight_value,avg_basket_value,n_cancellations,n_fail_delivery,first_order_date,last_order_date,cancellation_rate,failure_rate
0,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,1,124.99,124.99,0.0,0.0,0.0,21.88,124.99,0,0,2017-05-16 15:05:35,2017-05-16 15:05:35,0.0,0.0
1,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,1,289.00,289.00,0.0,0.0,0.0,46.48,289.00,0,0,2018-01-12 20:48:24,2018-01-12 20:48:24,0.0,0.0
2,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1,139.94,139.94,0.0,0.0,0.0,17.79,139.94,0,0,2018-05-19 16:07:45,2018-05-19 16:07:45,0.0,0.0


**d. Final checks**

In [46]:
# PROVIDE INFORMATION ABOUT THE NUMBER OF COLUMNS AND ROWS IN THE DATA FRAME

print(f'The df_items data frame contains {df_customers.shape[1]} columns, {df_customers.shape[0]} entries, and {df_customers.duplicated().sum()} duplicates.')


The df_items data frame contains 18 columns, 96096 entries, and 0 duplicates.


In [47]:
# INSPECT DATAFRAME MEDTADATA INCLUDING ENTRIES, COLUMNS, NON-NULL COUNTS, AND DATA TYPES.

df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   customer_unique_id             96096 non-null  object        
 1   customer_zip_code_prefix       96096 non-null  int64         
 2   customer_city                  96096 non-null  object        
 3   customer_state                 96096 non-null  object        
 4   n_orders                       96096 non-null  int64         
 5   total_price_bcanc_and_fail     96096 non-null  float64       
 6   total_price_aft_canc_and_fail  96096 non-null  float64       
 7   total_price_loss_canc          96096 non-null  float64       
 8   total_price_loss_fail          96096 non-null  float64       
 9   total_price_loss               96096 non-null  float64       
 10  total_freight_value            96096 non-null  float64       
 11  avg_basket_valu